# OSeMOSYS — Notebook de pruebas paso a paso

Construimos el flujo basico bloque a bloque.

**Estado actual:** crea el `AbstractModel` y prepara los CSVs de entrada (desde Excel SAND o desde una carpeta CSV existente). Las siguientes etapas (build_instance -> solve -> process_results) se iran agregando.

## 1. Bootstrap del entorno

Este notebook vive en `backend/app/simulation/`. Para que los imports `from app.` funcionen necesitamos:

1. Tener `backend/` en `sys.path`.
2. Posicionar el cwd en `backend/` (varios modulos asumen rutas relativas: `tmp/`, `.env`, etc.).

In [1]:
import os, sys
from pathlib import Path

# El notebook esta en backend/app/simulation/  ->  backend/ esta 2 niveles arriba
BACKEND_DIR = Path.cwd()
while BACKEND_DIR.name != "backend" and BACKEND_DIR.parent != BACKEND_DIR:
    BACKEND_DIR = BACKEND_DIR.parent
assert BACKEND_DIR.name == "backend", f"No encontre backend/ subiendo desde {Path.cwd()}"
REPO_ROOT = BACKEND_DIR.parent

if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))
os.chdir(BACKEND_DIR)

# Licencia Gurobi WLS: debe setearse ANTES de cualquier `import gurobipy`.
# Si gurobipy ya esta cargado en este kernel con otra licencia, hay que reiniciar.
_grb_lic = REPO_ROOT / "secrets" / "gurobi.lic"
if _grb_lic.is_file():
    os.environ["GRB_LICENSE_FILE"] = str(_grb_lic)

# Carga .env si existe (no es obligatorio para crear el modelo abstracto)
try:
    from dotenv import load_dotenv
    for env_file in (".env", ".env.local"):
        if (BACKEND_DIR / env_file).exists():
            load_dotenv(BACKEND_DIR / env_file)
            break
except ImportError:
    pass

print("cwd              :", os.getcwd())
print("backend          :", BACKEND_DIR)
print("GRB_LICENSE_FILE :", os.environ.get("GRB_LICENSE_FILE", "(no set)"))

cwd              : /Users/davidbedoya0/Documents/UPME/Demanda/Repositorio/OSEMOSYS_FINAL_2/Osemosys_UPME/backend
backend          : /Users/davidbedoya0/Documents/UPME/Demanda/Repositorio/OSEMOSYS_FINAL_2/Osemosys_UPME/backend
GRB_LICENSE_FILE : /Users/davidbedoya0/Documents/UPME/Demanda/Repositorio/OSEMOSYS_FINAL_2/Osemosys_UPME/secrets/gurobi.lic


## 2. Crear el modelo abstracto

`create_abstract_model(has_storage, has_udc)` define todos los sets, parametros, variables, restricciones y la funcion objetivo de OSeMOSYS, **sin datos**. El resultado es un `pyomo.environ.AbstractModel` listo para `create_instance(data)`.

Por ahora lo creamos con flags por defecto. En la seccion 4 lo regeneraremos con `has_storage` / `has_udc` coherentes con los datos cargados.

In [2]:
from app.simulation.core.model_definition import create_abstract_model

model = create_abstract_model(has_storage=False, has_udc=False)
model

## 3. Inspeccion rapida

Confirmamos que el modelo abstracto tiene los componentes esperados (Sets, Params, Vars, Constraints, Objective).

In [3]:
from pyomo.environ import Set, Param, Var, Constraint, Objective

def _count(component_type):
    return sum(1 for _ in model.component_objects(component_type, active=True))

print(f"Sets        : {_count(Set)}")
print(f"Params      : {_count(Param)}")
print(f"Vars        : {_count(Var)}")
print(f"Constraints : {_count(Constraint)}")
print(f"Objectives  : {_count(Objective)}")

Sets        : 9
Params      : 50
Vars        : 30
Constraints : 44
Objectives  : 1


## 4. Preparar CSVs de entrada

Hay dos caminos para llegar al directorio de CSVs que consume `build_instance`:

- **A) Desde un Excel SAND** (`run_data_processing_from_excel`) - genera los CSVs en una carpeta nueva, normaliza, completa matrices y desactiva UDC (modo Excel).
- **B) Desde una carpeta de CSVs ya existente** (`get_processing_result_from_csv_dir`) - solo lee los sets para construir el `ProcessingResult`; asume que los CSVs ya estan procesados.

Ambos caminos devuelven un `ProcessingResult` con `has_storage` / `has_udc` y los lookups id<->nombre. Esos flags se usaran en la siguiente celda para regenerar el `AbstractModel` con la configuracion correcta.

In [4]:
# Configura aqui la fuente de datos. Deja UNA de las dos opciones y comenta la otra.

REPO_ROOT = BACKEND_DIR.parent

# --- Opcion A: Excel SAND ----------------------------------------------------
EXCEL_PATH: Path | None = REPO_ROOT / "SAND_integrado_PA_MR_20_04.xlsx"
EXCEL_PATH: Path | None = REPO_ROOT / "29_04_26_SAND_INTEGRADO_CN.xlsx"
CSV_DIR_OUT: Path = BACKEND_DIR / "tmp" / "run_notebook_csv"   # destino de los CSVs generados

# --- Opcion B: carpeta de CSVs ya existente ---------------------------------
CSV_DIR_IN: Path | None = None              # p. ej. REPO_ROOT / "CSV"

# Sanity check: exactamente una opcion activa
assert (EXCEL_PATH is None) ^ (CSV_DIR_IN is None), \
    "Define exactamente una fuente: EXCEL_PATH o CSV_DIR_IN (la otra debe ser None)."

print("EXCEL_PATH :", EXCEL_PATH)
print("CSV_DIR_IN :", CSV_DIR_IN)
print("CSV_DIR_OUT:", CSV_DIR_OUT)

EXCEL_PATH : /Users/davidbedoya0/Documents/UPME/Demanda/Repositorio/OSEMOSYS_FINAL_2/Osemosys_UPME/29_04_26_SAND_INTEGRADO_CN.xlsx
CSV_DIR_IN : None
CSV_DIR_OUT: /Users/davidbedoya0/Documents/UPME/Demanda/Repositorio/OSEMOSYS_FINAL_2/Osemosys_UPME/backend/tmp/run_notebook_csv


In [5]:
import time
from app.simulation.core.data_processing import (
    run_data_processing_from_excel,
    get_processing_result_from_csv_dir,
)

if EXCEL_PATH is not None:
    assert EXCEL_PATH.is_file(), f"No existe el Excel: {EXCEL_PATH}"
    CSV_DIR_OUT.mkdir(parents=True, exist_ok=True)
    _t0 = time.perf_counter()
    processing_raw = run_data_processing_from_excel(
        excel_path=EXCEL_PATH,
        csv_dir=str(CSV_DIR_OUT),
        sheet_name="Parameters",
        div=1,
    )
    t_raw_seconds = time.perf_counter() - _t0
    csv_dir_raw = CSV_DIR_OUT
else:
    assert CSV_DIR_IN is not None and CSV_DIR_IN.is_dir(), f"No existe la carpeta CSV: {CSV_DIR_IN}"
    processing_raw = get_processing_result_from_csv_dir(str(CSV_DIR_IN))
    csv_dir_raw = CSV_DIR_IN
    t_raw_seconds = None

# Alias para celdas posteriores (hasta que la seccion 5/6 los reemplace por la version filtrada)
csv_dir = csv_dir_raw
processing = processing_raw

print(f"csv_dir_raw   : {csv_dir_raw}")
print(f"has_storage   : {processing_raw.has_storage}")
print(f"has_udc       : {processing_raw.has_udc}")
print(f"sets          : {len(processing_raw.sets)}")
print(f"#REGION       : {len(processing_raw.sets.get('REGION', []))}")
print(f"#TECHNOLOGY   : {len(processing_raw.sets.get('TECHNOLOGY', []))}")
print(f"#FUEL         : {len(processing_raw.sets.get('FUEL', []))}")
print(f"#YEAR         : {len(processing_raw.sets.get('YEAR', []))}")
if t_raw_seconds is not None:
    print(f"Tiempo gen raw: {t_raw_seconds:.2f} s")

  Leyendo Excel SAND: /Users/davidbedoya0/Documents/UPME/Demanda/Repositorio/OSEMOSYS_FINAL_2/Osemosys_UPME/29_04_26_SAND_INTEGRADO_CN.xlsx
  Parámetros encontrados: 38
  Generando sets...
  Generando parámetros...
  Filtrando parámetros por sets...
  Completando matrices (ActivityRatio)...
  Completando matriz (EmissionActivityRatio)...
  Completando matriz (VariableCost)...
  Procesando emisiones a la entrada...
  Total CSVs generados: 45
csv_dir_raw   : /Users/davidbedoya0/Documents/UPME/Demanda/Repositorio/OSEMOSYS_FINAL_2/Osemosys_UPME/backend/tmp/run_notebook_csv
has_storage   : False
has_udc       : False
sets          : 7
#REGION       : 1
#TECHNOLOGY   : 408
#FUEL         : 115
#YEAR         : 33
Tiempo gen raw: 28.15 s


In [6]:
# Listado rapido de los CSVs disponibles en csv_dir
csv_files = sorted(p.name for p in csv_dir.glob("*.csv"))
print(f"Total CSVs: {len(csv_files)}")
for name in csv_files:
    print("  -", name)

Total CSVs: 45
  - AccumulatedAnnualDemand.csv
  - AnnualEmissionLimit.csv
  - AnnualExogenousEmission.csv
  - AvailabilityFactor.csv
  - CapacityFactor.csv
  - CapacityOfOneTechnologyUnit.csv
  - CapacityToActivityUnit.csv
  - CapitalCost.csv
  - DepreciationMethod.csv
  - DiscountRate.csv
  - EMISSION.csv
  - EmissionActivityRatio.csv
  - EmissionsPenalty.csv
  - FUEL.csv
  - FixedCost.csv
  - InputActivityRatio.csv
  - MODE_OF_OPERATION.csv
  - ModelPeriodEmissionLimit.csv
  - ModelPeriodExogenousEmission.csv
  - OperationalLife.csv
  - OutputActivityRatio.csv
  - REGION.csv
  - REMinProductionTarget.csv
  - RETagFuel.csv
  - RETagTechnology.csv
  - ReserveMargin.csv
  - ReserveMarginTagFuel.csv
  - ReserveMarginTagTechnology.csv
  - ResidualCapacity.csv
  - SpecifiedAnnualDemand.csv
  - SpecifiedDemandProfile.csv
  - TECHNOLOGY.csv
  - TIMESLICE.csv
  - TotalAnnualMaxCapacity.csv
  - TotalAnnualMaxCapacityInvestment.csv
  - TotalAnnualMinCapacity.csv
  - TotalAnnualMinCapacityInves

### Re-crear el modelo abstracto con los flags reales

Si en la seccion 2 fijamos `has_storage=False, has_udc=True` por defecto, ahora regeneramos el modelo coherente con los datos cargados (en modo Excel `has_udc` queda en `False` automaticamente).

In [7]:
model = create_abstract_model(
    has_storage=processing.has_storage,
    has_udc=processing.has_udc,
)
print(f"AbstractModel recreado con has_storage={processing.has_storage}, has_udc={processing.has_udc}")

AbstractModel recreado con has_storage=False, has_udc=False


## 5. Filtrar CSVs por defaults (post-generacion)

Filtramos **despues** de que `run_data_processing_from_excel` haya producido `csv_dir_raw`. La idea es:

1. **Sets, post-procesado y estructura quedan intactos** (vienen de la pipeline original sin tocar).
2. Para cada CSV de **parametro** con default escalar conocido, eliminamos las filas donde `VALUE == default` (el modelo aplicara el mismo default automaticamente).
3. Los CSVs de **sets** (`REGION.csv`, `TECHNOLOGY.csv`, etc.) y todo lo demas se copia sin cambios.

Resultado: dos directorios persistentes con la misma estructura logica pero diferente cantidad de filas en los CSVs de parametros.

> Esto evita los problemas de derivar sets desde un SAND filtrado: ya no hace falta tocar el flujo SAND -> CSV.

In [8]:
import shutil
import numpy as np
import pandas as pd
from pathlib import Path
from pyomo.environ import Param

_NO_VALUE = Param.NoValue


def get_model_defaults(abstract_model) -> dict[str, object]:
    """Devuelve {param_name: default_or_None} desde un AbstractModel Pyomo.

    None significa: el Param no declara default (Pyomo exigira valor explicito
    para cada indice). Los defaults callable se devuelven tal cual.
    """
    out: dict[str, object] = {}
    for p in abstract_model.component_objects(Param, active=True):
        v = getattr(p, "_default_val", _NO_VALUE)
        out[p.name] = None if v is _NO_VALUE else v
    return out


def filter_csv_by_defaults(
    src_dir: Path,
    dst_dir: Path,
    defaults: dict,
    *,
    atol: float = 1e-9,
) -> dict[str, tuple[int, int]]:
    """Copia src_dir -> dst_dir filtrando filas con VALUE == default en CSVs de parametros.

    - CSVs cuyo nombre no esta en `defaults` (sets, archivos auxiliares) se copian tal cual.
    - CSVs de parametros con default escalar conocido: se eliminan las filas donde
      `VALUE` coincide numericamente con el default (tolerancia `atol`).
    - CSVs de parametros con default callable o sin default declarado: se copian tal cual.

    Returns
    -------
    dict {filename: (rows_in, rows_out)} solo para los CSVs efectivamente filtrados.
    """
    src_dir = Path(src_dir)
    dst_dir = Path(dst_dir)
    if dst_dir.exists():
        shutil.rmtree(dst_dir)
    dst_dir.mkdir(parents=True, exist_ok=True)

    report: dict[str, tuple[int, int]] = {}
    for src in sorted(src_dir.glob("*.csv")):
        param = src.stem
        default = defaults.get(param)
        dst = dst_dir / src.name

        if default is None or not isinstance(default, (int, float, np.integer, np.floating)):
            shutil.copy(src, dst)
            continue

        df = pd.read_csv(src)
        rows_in = len(df)
        if "VALUE" not in df.columns or rows_in == 0:
            shutil.copy(src, dst)
            continue

        vals = pd.to_numeric(df["VALUE"], errors="coerce").astype(float)
        is_default = np.isclose(vals, float(default), atol=atol, equal_nan=False)
        df_out = df.loc[~is_default]
        df_out.to_csv(dst, index=False)
        report[src.name] = (rows_in, len(df_out))

    return report

In [9]:
import time
from app.simulation.core.data_processing import get_processing_result_from_csv_dir

CSV_DIR_OUT_FILTERED = BACKEND_DIR / "tmp" / "run_notebook_csv_filtered"

defaults = get_model_defaults(model)
n_with = sum(1 for v in defaults.values() if v is not None)
print(f"Defaults declarados: {n_with}/{len(defaults)} parametros")

_t0 = time.perf_counter()
csv_filter_report = filter_csv_by_defaults(csv_dir_raw, CSV_DIR_OUT_FILTERED, defaults)
t_filtered_seconds = time.perf_counter() - _t0

processing_filtered = get_processing_result_from_csv_dir(str(CSV_DIR_OUT_FILTERED))
csv_dir_filtered = CSV_DIR_OUT_FILTERED

# Resumen
total_in  = sum(r[0] for r in csv_filter_report.values())
total_out = sum(r[1] for r in csv_filter_report.values())
print()
print(f"CSVs procesados (con default escalar): {len(csv_filter_report)}")
print(f"  filas totales antes  : {total_in}")
print(f"  filas totales despues: {total_out}")
print(f"  reduccion            : {total_in - total_out} filas ({(1 - total_out/total_in)*100:.1f}%)" if total_in else "  (nada que filtrar)")
print(f"\nTiempo de filtrado     : {t_filtered_seconds:.3f} s")
print(f"csv_dir_filtered       : {csv_dir_filtered}")
print(f"  has_storage          : {processing_filtered.has_storage}")
print(f"  has_udc              : {processing_filtered.has_udc}")
print(f"  #TECHNOLOGY          : {len(processing_filtered.sets.get('TECHNOLOGY', []))}")
print(f"  #FUEL                : {len(processing_filtered.sets.get('FUEL', []))}")
print(f"  #TIMESLICE           : {len(processing_filtered.sets.get('TIMESLICE', []))}")

Defaults declarados: 46/50 parametros

CSVs procesados (con default escalar): 36
  filas totales antes  : 721716
  filas totales despues: 163078
  reduccion            : 558638 filas (77.4%)

Tiempo de filtrado     : 0.335 s
csv_dir_filtered       : /Users/davidbedoya0/Documents/UPME/Demanda/Repositorio/OSEMOSYS_FINAL_2/Osemosys_UPME/backend/tmp/run_notebook_csv_filtered
  has_storage          : False
  has_udc              : False
  #TECHNOLOGY          : 408
  #FUEL                : 115
  #TIMESLICE           : 1


In [10]:
# Detalle por CSV: top reducciones absolutas
import pandas as pd

detail = pd.DataFrame(
    [(name, ri, ro, ri - ro, (1 - ro/ri) if ri else 0.0)
     for name, (ri, ro) in csv_filter_report.items()],
    columns=["csv", "rows_in", "rows_out", "delta", "pct_drop"],
).sort_values("delta", ascending=False)

print("Top 20 CSVs con mayor reduccion de filas:")
print(f"  {'csv':<35s} {'rows_in':>10s} {'rows_out':>10s} {'delta':>10s} {'%':>8s}")
for _, r in detail.head(20).iterrows():
    print(f"  {r['csv']:<35s} {r['rows_in']:>10d} {r['rows_out']:>10d} {r['delta']:>10d} {r['pct_drop']*100:>7.1f}%")

# Sanity: csvs que quedan vacios
empty_after = detail[detail["rows_out"] == 0]
if len(empty_after):
    print(f"\nCSVs que quedaron VACIOS despues del filtro ({len(empty_after)}):")
    for _, r in empty_after.iterrows():
        print(f"  {r['csv']:<35s} (eran {r['rows_in']} filas, todas == default)")

Top 20 CSVs con mayor reduccion de filas:
  csv                                    rows_in   rows_out      delta        %
  OutputActivityRatio.csv                 352968      13695     339273    96.1%
  EmissionActivityRatio.csv               141900      43692      98208    69.2%
  AvailabilityFactor.csv                   13464          0      13464   100.0%
  CapacityOfOneTechnologyUnit.csv          13332          0      13332   100.0%
  RETagTechnology.csv                      13332          0      13332   100.0%
  TotalAnnualMinCapacity.csv               13332         54      13278    99.6%
  TotalAnnualMinCapacityInvestment.csv      13332        176      13156    98.7%
  ReserveMarginTagTechnology.csv           13332        825      12507    93.8%
  CapacityFactor.csv                       12870        825      12045    93.6%
  TotalTechnologyAnnualActivityLowerLimit.csv      13431       2334      11097    82.6%
  ResidualCapacity.csv                     13464       7611       585

## 6. Comparativa raw vs filtrado

Despues de correr secciones 4 (raw) y 5 (filtrado) tenemos **dos directorios de CSVs persistentes**:

- `csv_dir_raw` -> `tmp/run_notebook_csv/` (generado por `run_data_processing_from_excel`)
- `csv_dir_filtered` -> `tmp/run_notebook_csv_filtered/` (copia con filas-default removidas en CSVs de parametros)

Ambos siguen disponibles para que la seccion 7 (build_instance) pueda compararlos.

La celda de abajo reporta:

- **Tiempos**: `t_raw_seconds` (generacion completa SAND -> CSVs) vs `t_filtered_seconds` (solo el filtrado, no comparable directamente).
- **Tamano total** en bytes y numero total de filas.
- **Top CSVs con mayor reduccion** (donde el filtrado mas comprimio).
- Sanity check de archivos exclusivos a uno u otro (no deberia haber, ya que el filtrado solo afecta filas, no archivos).

> Nota: la diferencia interesante en tiempos esta en la seccion 7 (DataPortal/build_instance), no aqui.

In [11]:
def _csv_summary(directory: Path) -> dict:
    files = sorted(p for p in Path(directory).glob("*.csv"))
    total_bytes = 0
    total_rows = 0
    per_file: dict[str, tuple[int, int]] = {}  # name -> (rows, bytes)
    for p in files:
        size = p.stat().st_size
        with p.open("r", encoding="utf-8") as f:
            rows = max(0, sum(1 for _ in f) - 1)  # menos el header
        per_file[p.name] = (rows, size)
        total_rows += rows
        total_bytes += size
    return {
        "n_files": len(files),
        "total_bytes": total_bytes,
        "total_rows": total_rows,
        "per_file": per_file,
    }


raw_sum = _csv_summary(csv_dir_raw)
flt_sum = _csv_summary(csv_dir_filtered)


def _ratio(a, b):
    return f"{a / b:.2f}x" if b else "n/a"


print(f"{'':<22s} {'RAW':>14s} {'FILTRADO':>14s} {'ratio':>10s}")
print("-" * 64)
print(f"{'Tiempo gen [s]':<22s} {t_raw_seconds:>14.2f} {t_filtered_seconds:>14.2f} {_ratio(t_filtered_seconds, t_raw_seconds):>10s}")
print(f"{'# CSVs':<22s} {raw_sum['n_files']:>14d} {flt_sum['n_files']:>14d} {_ratio(flt_sum['n_files'], raw_sum['n_files']):>10s}")
print(f"{'Total bytes':<22s} {raw_sum['total_bytes']:>14d} {flt_sum['total_bytes']:>14d} {_ratio(flt_sum['total_bytes'], raw_sum['total_bytes']):>10s}")
print(f"{'Total filas':<22s} {raw_sum['total_rows']:>14d} {flt_sum['total_rows']:>14d} {_ratio(flt_sum['total_rows'], raw_sum['total_rows']):>10s}")

# Diferencias por archivo
all_files = set(raw_sum["per_file"]) | set(flt_sum["per_file"])
deltas = []
for f in all_files:
    r_rows, r_bytes = raw_sum["per_file"].get(f, (0, 0))
    f_rows, f_bytes = flt_sum["per_file"].get(f, (0, 0))
    deltas.append((f, r_rows, f_rows, r_rows - f_rows))

deltas_sorted = sorted(deltas, key=lambda x: x[3], reverse=True)

print(f"\nTop 15 CSVs con mayor reduccion de filas:")
print(f"  {'archivo':<35s} {'raw':>10s} {'flt':>10s} {'delta':>10s}")
for name, r, f, d in deltas_sorted[:15]:
    print(f"  {name:<35s} {r:>10d} {f:>10d} {d:>10d}")

# CSVs que aparecen en uno pero no en el otro (sanity check)
only_raw = sorted(set(raw_sum["per_file"]) - set(flt_sum["per_file"]))
only_flt = sorted(set(flt_sum["per_file"]) - set(raw_sum["per_file"]))
if only_raw:
    print(f"\nSolo en RAW ({len(only_raw)}): {only_raw}")
if only_flt:
    print(f"Solo en FILTRADO ({len(only_flt)}): {only_flt}")

                                  RAW       FILTRADO      ratio
----------------------------------------------------------------
Tiempo gen [s]                  28.15           0.33      0.01x
# CSVs                             45             45      1.00x
Total bytes                  26935198        6625539      0.25x
Total filas                    725058         166420      0.23x

Top 15 CSVs con mayor reduccion de filas:
  archivo                                    raw        flt      delta
  OutputActivityRatio.csv                 352968      13695     339273
  EmissionActivityRatio.csv               141900      43692      98208
  AvailabilityFactor.csv                   13464          0      13464
  CapacityOfOneTechnologyUnit.csv          13332          0      13332
  RETagTechnology.csv                      13332          0      13332
  TotalAnnualMinCapacity.csv               13332         54      13278
  TotalAnnualMinCapacityInvestment.csv      13332        176      13156
  R

## 7. Construir la instancia (build_instance) — comparativa raw vs filtrado

Usamos la funcion oficial del repo:

```python
from app.simulation.core.instance_builder import build_instance
build_instance(model, csv_dir, has_storage=..., has_udc=...) -> ConcreteModel
```

Internamente carga via Pyomo `DataPortal` y crea una `ConcreteModel`. Es donde se nota el efecto del filtrado: menos filas en los CSVs -> menos parsing -> menos elementos a indexar en parametros.

Construimos **dos instancias independientes** (cada una con su propio `AbstractModel` para evitar estado compartido) y medimos:

- Tiempo de DataPortal + `create_instance`.
- Numero de variables, restricciones y elementos parametrizados de cada instancia (deberian ser **iguales**: el filtrado no cambia el problema, solo evita que el mismo valor se cargue como override).

In [12]:
import time
from app.simulation.core.instance_builder import build_instance

# Dos AbstractModel independientes (mismo flags, pero evitamos compartir estado)
model_raw = create_abstract_model(
    has_storage=processing_raw.has_storage,
    has_udc=processing_raw.has_udc,
)
model_flt = create_abstract_model(
    has_storage=processing_filtered.has_storage,
    has_udc=processing_filtered.has_udc,
)

print("Construyendo instancia RAW...")
_t0 = time.perf_counter()
instance_raw = build_instance(
    model_raw,
    str(csv_dir_raw),
    has_storage=processing_raw.has_storage,
    has_udc=processing_raw.has_udc,
)
t_build_raw = time.perf_counter() - _t0
print(f"  -> {t_build_raw:.2f} s")

print("Construyendo instancia FILTRADO...")
_t0 = time.perf_counter()
instance_flt = build_instance(
    model_flt,
    str(csv_dir_filtered),
    has_storage=processing_filtered.has_storage,
    has_udc=processing_filtered.has_udc,
)
t_build_flt = time.perf_counter() - _t0
print(f"  -> {t_build_flt:.2f} s")

Construyendo instancia RAW...
           0 seconds to construct Set YEAR; 1 index total
           0 seconds to construct Set TECHNOLOGY; 1 index total
           0 seconds to construct Set TIMESLICE; 1 index total
           0 seconds to construct Set FUEL; 1 index total
           0 seconds to construct Set EMISSION; 1 index total
           0 seconds to construct Set MODE_OF_OPERATION; 1 index total
           0 seconds to construct Set REGION; 1 index total
           0 seconds to construct Set FLEXIBLEDEMANDTYPE; 1 index total
           0 seconds to construct Set UDC; 1 index total
           0 seconds to construct Set SetProduct_OrderedSet; 1 index total
           0 seconds to construct Param YearSplit; 33 indices total
           0 seconds to construct Param DiscountRate; 1 index total
           0 seconds to construct Set SetProduct_OrderedSet; 1 index total
           0 seconds to construct Param DiscountRateIdv; 408 indices total
           0 seconds to construct Set SetPro

In [13]:
def _instance_summary(instance) -> dict:
    n_vars = sum(len(v) for v in instance.component_objects(Var, active=True))
    n_cons = sum(len(c) for c in instance.component_objects(Constraint, active=True))
    n_param_elems = sum(len(p) for p in instance.component_objects(Param, active=True))
    n_set_elems = sum(len(s) for s in instance.component_objects(Set, active=True))
    return {
        "vars": n_vars,
        "cons": n_cons,
        "param_elems": n_param_elems,
        "set_elems": n_set_elems,
    }


sum_raw = _instance_summary(instance_raw)
sum_flt = _instance_summary(instance_flt)


def _ratio(a, b):
    return f"{a / b:.2f}x" if b else "n/a"


print(f"{'':<22s} {'RAW':>14s} {'FILTRADO':>14s} {'ratio':>10s}")
print("-" * 64)
print(f"{'Tiempo build [s]':<22s} {t_build_raw:>14.2f} {t_build_flt:>14.2f} {_ratio(t_build_flt, t_build_raw):>10s}")
print(f"{'# Variables':<22s} {sum_raw['vars']:>14d} {sum_flt['vars']:>14d} {_ratio(sum_flt['vars'], sum_raw['vars']):>10s}")
print(f"{'# Restricciones':<22s} {sum_raw['cons']:>14d} {sum_flt['cons']:>14d} {_ratio(sum_flt['cons'], sum_raw['cons']):>10s}")
print(f"{'# Param elements':<22s} {sum_raw['param_elems']:>14d} {sum_flt['param_elems']:>14d} {_ratio(sum_flt['param_elems'], sum_raw['param_elems']):>10s}")
print(f"{'# Set elements':<22s} {sum_raw['set_elems']:>14d} {sum_flt['set_elems']:>14d} {_ratio(sum_flt['set_elems'], sum_raw['set_elems']):>10s}")

# Sanidad: si vars/cons difieren, el filtrado cambio el problema (NO deberia pasar)
if sum_raw["vars"] != sum_flt["vars"] or sum_raw["cons"] != sum_flt["cons"]:
    print("\n!!  ALERTA: las instancias difieren en variables o restricciones.")
    print("    El filtrado NO deberia cambiar el modelo. Revisa el reporte de la seccion 5.")
else:
    print("\nOK: ambas instancias tienen el mismo numero de variables y restricciones.")

# Tiempo total acumulado (gen CSV + build instance) por flujo
print(f"\nTiempo total (gen+build):")
print(f"  RAW      : {t_raw_seconds + t_build_raw:.2f} s")
print(f"  FILTRADO : {t_filtered_seconds + t_build_flt:.2f} s")

                                  RAW       FILTRADO      ratio
----------------------------------------------------------------
Tiempo build [s]                25.75          25.88      1.00x
# Variables                    701075         701075      1.00x
# Restricciones                777076         777076      1.00x
# Param elements              6773118        6773118      1.00x
# Set elements                    570            570      1.00x

OK: ambas instancias tienen el mismo numero de variables y restricciones.

Tiempo total (gen+build):
  RAW      : 53.90 s
  FILTRADO : 26.21 s


In [14]:
def _approx_equal(a, b, atol=1e-9):
    if a is None or b is None:
        return a is b
    try:
        return np.isclose(float(a), float(b), atol=atol)
    except (TypeError, ValueError):
        return a == b


def compare_param_data(inst_raw, inst_flt, defaults, *, atol=1e-9):
    """Verifica que ambas instancias tengan los mismos VALORES efectivos para cada Param.

    Reglas:
      - flt_keys debe ser subconjunto de raw_keys (el filtrado solo borra filas).
      - Para keys compartidas: valores deben coincidir (el filtrado no debe alterar valores).
      - Para keys solo en raw: el valor en raw debe ser el default (por eso se filtro).

    Usa Param.extract_values() para obtener valores numericos (compatible con Param mutable).

    Devuelve (summary_df, ok).
    """
    rows = []
    ok_global = True

    for p_raw in inst_raw.component_objects(Param, active=True):
        name = p_raw.name
        p_flt = getattr(inst_flt, name, None)
        if p_flt is None:
            rows.append((name, "MISSING_IN_FLT", 0, 0, 0, 0, 0))
            ok_global = False
            continue

        raw_dict = p_raw.extract_values()
        flt_dict = p_flt.extract_values()
        raw_keys = set(raw_dict.keys())
        flt_keys = set(flt_dict.keys())

        # 1) flt - raw: orphan keys (no deberian existir)
        orphan = flt_keys - raw_keys
        # 2) shared: mismatch en valores
        shared = raw_keys & flt_keys
        n_mismatch = sum(1 for k in shared if not _approx_equal(raw_dict[k], flt_dict[k], atol))
        # 3) raw - flt: deberian ser default
        raw_only = raw_keys - flt_keys
        d = defaults.get(name)
        if d is not None and isinstance(d, (int, float, np.integer, np.floating)):
            n_not_default = sum(1 for k in raw_only if not _approx_equal(raw_dict[k], d, atol))
        else:
            # Sin default escalar conocido; raw_only no deberia existir (no filtramos ese param)
            n_not_default = len(raw_only)

        status = "OK" if (not orphan and n_mismatch == 0 and n_not_default == 0) else "FAIL"
        if status == "FAIL":
            ok_global = False
        rows.append((name, status, len(raw_keys), len(flt_keys), len(orphan), n_mismatch, n_not_default))

    summary_df = pd.DataFrame(
        rows,
        columns=["param", "status", "raw_keys", "flt_keys", "orphan_in_flt", "value_mismatch", "raw_only_not_default"],
    )
    return summary_df, ok_global


cmp_df, all_ok = compare_param_data(instance_raw, instance_flt, defaults)

print(f"{'param':<40s} {'status':>6s} {'raw':>10s} {'flt':>10s} {'orphan':>8s} {'mism':>6s} {'!=def':>6s}")
print("-" * 92)
for _, r in cmp_df.iterrows():
    flag = "  " if r["status"] == "OK" else "!!"
    print(f"{flag} {r['param']:<37s} {r['status']:>6s} {r['raw_keys']:>10d} {r['flt_keys']:>10d} {r['orphan_in_flt']:>8d} {r['value_mismatch']:>6d} {r['raw_only_not_default']:>6d}")

print()
if all_ok:
    print("OK: ambos modelos contienen los MISMOS datos efectivos.")
    print("    (cada key omitida en flt corresponde a un valor == default en raw)")
else:
    n_fail = (cmp_df["status"] != "OK").sum()
    print(f"!! ALERTA: {n_fail} parametros con discrepancias.")
    print("   Revisa columnas orphan_in_flt / value_mismatch / raw_only_not_default.")

param                                    status        raw        flt   orphan   mism  !=def
--------------------------------------------------------------------------------------------
   YearSplit                                 OK         33         33        0      0      0
   DiscountRate                              OK          1          1        0      0      0
   DiscountRateIdv                           OK        408        408        0      0      0
   OperationalLife                           OK        408        408        0      0      0
   CapitalRecoveryFactor                     OK        408        408        0      0      0
   PvAnnuity                                 OK        408        408        0      0      0
   DepreciationMethod                        OK          1          1        0      0      0
   AccumulatedAnnualDemand                   OK       3795       3795        0      0      0
   SpecifiedAnnualDemand                     OK       3795       3795 

## 8. Resolver con distintos solvers (Gurobi, HiGHS, GLPK)

No usamos `solve_model` del repo; configuramos cada solver explicitamente para poder ajustar opciones y diagnosticar problemas de licencia.

**Sobre Gurobi:** el `gurobipy` que viene por pip incluye una **licencia Restricted** (limite ~2000 vars/constraints). Para un modelo OSeMOSYS real necesitas una licencia completa:

- Academic: `grbgetkey <KEY>` (token nominal por maquina), o
- Configurar `GRB_LICENSE_FILE=/ruta/gurobi.lic` antes de arrancar Jupyter.

Si la licencia esta limitada, Gurobi fallara con `Model too large for size-limited license`. La celda de diagnostico abajo te dira que tienes.

In [15]:
import os, shutil
import pyomo.environ as pyo  # registra los plugins de solvers


In [ ]:

print("=== Solvers disponibles via Pyomo ===")
solver_names = ["gurobi", "gurobi_direct", "gurobi_persistent", "appsi_highs", 'highs', "glpk", "cbc"]
solver_status = {}
for name in solver_names:
    try:
        s = pyo.SolverFactory(name)
        ok = bool(s.available(exception_flag=False))
    except Exception as e:
        ok = False
        print(f"  {name:<22s} ERROR: {type(e).__name__}: {e}")
        continue
    solver_status[name] = ok
    print(f"  {name:<22s} available={ok}")

print()
print("=== Diagnostico Gurobi ===")
try:
    import gurobipy as gp
    print(f"  gurobipy: {gp.gurobi.version()}")
    print(f"  GRB_LICENSE_FILE: {os.environ.get('GRB_LICENSE_FILE', '(no set)')}")
    print(f"  GUROBI_HOME     : {os.environ.get('GUROBI_HOME', '(no set)')}")
    print(f"  gurobi_cl en PATH: {shutil.which('gurobi_cl')}")
    # Test de licencia con Env+Model explicitos y dispose para no bloquear el slot WLS
    env = None
    m = None
    try:
        env = gp.Env()
        m = gp.Model("license_test", env=env)
        m.setParam("OutputFlag", 0)
        N = 5000
        xs = m.addVars(N, name="x")
        m.setObjective(gp.quicksum(xs[i] for i in range(N)), gp.GRB.MINIMIZE)
        for i in range(N):
            m.addConstr(xs[i] >= 1)
        m.optimize()
        print(f"  test licencia ({N} vars): OK status={m.status}, obj={m.objVal:.2f}")
    except gp.GurobiError as e:
        print(f"  test licencia: ERROR -> {e}")
    finally:
        # Liberar recursos para que el slot single-use de WLS quede libre
        if m is not None:
            m.dispose()
        if env is not None:
            env.dispose()
except ImportError:
    print("  gurobipy no instalado")

print()
print("=== Diagnostico HiGHS ===")
try:
    import highspy
    print(f"  highspy: instalado")
except ImportError:
    print("  highspy no instalado")

print()
print("=== Diagnostico GLPK ===")
print(f"  glpsol: {shutil.which('glpsol')}")

In [16]:
import time
from pyomo.opt import SolverStatus, TerminationCondition

# =============================================================================
#  Configuracion de solvers
#  - Una entry por cada wrapper de Pyomo que quieras probar.
#  - "options" se pasa al solver. Vacio = usar defaults de fabrica.
#  - "io" / "notes" son metadatos; no afectan el solve.
# =============================================================================
SOLVER_CONFIGS: dict[str, dict] = {
    "gurobi": {
        "io": "shell o python (Pyomo elige segun gurobi_cl/gurobipy disponibles)",
        "notes": "Wrapper generico de Pyomo para Gurobi. En este setup -> gurobipy.",
        "options": {},
    },
    "gurobi_direct": {
        "io": "python (gurobipy)",
        "notes": "Forzado a usar gurobipy directo, sin LP intermedio.",
        "options": {},
    },
    "gurobi_persistent": {
        "io": "python (gurobipy persistent)",
        "notes": "Como direct, retiene el modelo para re-solves incrementales.",
        "options": {},
    },
    "appsi_highs": {
        "io": "python (highspy via APPSI)",
        "notes": "HiGHS via interface APPSI (persistente). Recomendado en Pyomo moderno.",
        "options": {},
    },
    "highs": {
        "io": "python (highspy)",
        "notes": "HiGHS via wrapper basico. Mismo solver subyacente que appsi_highs.",
        "options": {},
    },
    "glpk": {
        "io": "shell (LP file + glpsol)",
        "notes": "GLPK via glpsol. Suele quedarse corto en modelos OSeMOSYS grandes.",
        "options": {},
    },
}


# =============================================================================
#  Mapeo de parametros equivalentes entre solvers
#
#  Para alinear tolerancias entre solvers (futuro benchmark), basta con copiar
#  los valores deseados a SOLVER_CONFIGS[<solver>]["options"] usando la columna
#  correspondiente al wrapper.
# =============================================================================
PARAM_EQUIVALENTS = {
    # nombre logico         gurobi*                  highs (appsi/plain)                  glpk
    "feas_tol":            ("FeasibilityTol",        "primal_feasibility_tolerance",      "tol_bnd"),
    "opt_tol":             ("OptimalityTol",         "dual_feasibility_tolerance",        "tol_dj"),
    "barrier_conv_tol":    ("BarConvTol",            "ipm_optimality_tolerance",          None),
    "method":              ("Method",                "solver",                            "method"),
    "time_limit_s":        ("TimeLimit",             "time_limit",                        "tmlim"),
    "threads":             ("Threads",               "threads",                           None),
    "presolve":            ("Presolve",              "presolve",                          "presol"),
    "output":              ("OutputFlag",            "output_flag",                       None),
}

# Defaults de Gurobi - referencia para alinear los demas solvers en benchmarks.
GUROBI_DEFAULT_TOLERANCES = {
    "FeasibilityTol": 1e-6,
    "OptimalityTol":  1e-6,
    "BarConvTol":     1e-8,
}

# Ejemplo (NO aplicado por defecto). Para alinear HiGHS a Gurobi:
#   SOLVER_CONFIGS["appsi_highs"]["options"].update({
#       "primal_feasibility_tolerance": 1e-6,
#       "dual_feasibility_tolerance":   1e-6,
#       "ipm_optimality_tolerance":     1e-8,
#   })


def solve_with(instance, solver_name: str, *, tee: bool = True) -> dict:
    """Resuelve `instance` con el solver indicado. Devuelve dict con tiempos, status y objetivo."""
    cfg = SOLVER_CONFIGS.get(solver_name, {})
    solver = pyo.SolverFactory(solver_name)
    for k, v in cfg.get("options", {}).items():
        solver.options[k] = v

    t0 = time.perf_counter()
    try:
        results = solver.solve(instance, tee=tee, load_solutions=True)
        elapsed = time.perf_counter() - t0
        obj_val = None
        for o in instance.component_objects(pyo.Objective, active=True):
            obj_val = pyo.value(o)
            break
        return {
            "solver": solver_name,
            "elapsed_s": elapsed,
            "status": str(results.solver.status),
            "termination": str(results.solver.termination_condition),
            "objective": obj_val,
            "error": None,
        }
    except Exception as e:
        elapsed = time.perf_counter() - t0
        return {
            "solver": solver_name,
            "elapsed_s": elapsed,
            "status": "error",
            "termination": "error",
            "objective": None,
            "error": f"{type(e).__name__}: {e}",
        }

In [19]:
# Selecciona la instancia sobre la que resolver. instance_flt suele ser mas rapida
# (CSVs filtrados -> mismo modelo, pero menos parsing).
INSTANCE_TO_SOLVE = instance_flt   # cambia a instance_raw para comparar

# Solvers a intentar en orden. Comenta/descomenta segun lo que esta disponible
# (mira la celda de diagnostico). Los no disponibles se saltan automaticamente.
SOLVERS_TO_TRY = [
    "gurobi",              # generico (vez que escogio gurobipy en este setup)
    "gurobi_direct",     # gurobipy explicito
    # "gurobi_persistent", # gurobipy persistente
    "appsi_highs",
    "highs",
    "glpk",              # tipicamente OSeMOSYS es muy grande para GLPK
]

solve_results = []
for sname in SOLVERS_TO_TRY:
    # if not solver_status.get(sname, False):
    #     print(f"\n--- {sname}: NO DISPONIBLE, salteando ---")
    #     continue
    print(f"\n=========== Resolviendo con {sname} ===========")
    res = solve_with(INSTANCE_TO_SOLVE, sname, tee=True)
    print(f"\n  status      : {res['status']}")
    print(f"  termination : {res['termination']}")
    print(f"  elapsed_s   : {res['elapsed_s']:.2f}")
    print(f"  objective   : {res['objective']}")
    if res["error"]:
        print(f"  ERROR       : {res['error']}")
    solve_results.append(res)


=========== Resolviendo con gurobi ===========
Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2810970
WLS license 2810970 - registered to Unidad de Planeacion Minero Energetica

  status      : error
  termination : error
  elapsed_s   : 6.30
  objective   : None
  ERROR       : GurobiError: Single-use license. Another Gurobi process with pid 60453 running.

=========== Resolviendo con gurobi_direct ===========
Gurobi process with pid 60453 running.

    Set parameter WLSAccessID Set parameter WLSSecret Set parameter LicenseID
    to value 2810970 WLS license 2810970 - registered to Unidad de Planeacion
    Minero Energetica

  status      : error
  termination : error
  elapsed_s   : 0.00
  objective   : None
  ERROR       : ApplicationError: Could not create Model for <class 'pyomo.solvers.plugins.solvers.gurobi_direct.GurobiDirect'> solver plugin - gurobi message=Could not create Model - gurobi message=Single-use license. Another Gurobi process w

KeyboardInterrupt: 

In [25]:
# Resumen comparativo
if not solve_results:
    print("Nada que comparar (no se ejecuto ningun solver).")
else:
    df_solve = pd.DataFrame(solve_results)
    print(df_solve[["solver", "elapsed_s", "status", "termination", "objective", "error"]].to_string(index=False))

    # Verificacion de coincidencia de objetivo (ignorando NaN/error)
    objs = df_solve.loc[df_solve["objective"].notna(), "objective"].tolist()
    if len(objs) >= 2:
        diff_max = max(objs) - min(objs)
        rel = diff_max / abs(objs[0]) if objs[0] else 0
        print(f"\nDiferencia maxima en objetivo: {diff_max:.6e} (rel: {rel:.2e})")
        if rel < 1e-6:
            print("OK: los solvers coinciden en el optimo.")
        else:
            print("!! ALERTA: los solvers difieren en el optimo. Revisa tolerancias o presolve.")

     solver  elapsed_s status termination  objective                                                                                                                                                                                                                                                                                                                                                                                      error
     gurobi   6.474101  error       error        NaN                                                                                                                                                                                                                                                                                                            GurobiError: Single-use license. Another Gurobi process with pid 60453 running.
appsi_highs 108.127804  error       error        NaN RuntimeError: A feasible solution was not found, so no solution can be loaded. If u

# 8. Solve

In [16]:
instance_flt.write("instance.lp", io_options={"symbolic_solver_labels": True})


('instance.lp', 5142566896)

In [17]:
import gurobipy as gp 

m = gp.read("instance.lp")




Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2810970
WLS license 2810970 - registered to Unidad de Planeacion Minero Energetica


GurobiError: Single-use license. Another Gurobi process with pid 60453 running.

In [ ]:
solver = SolverFactory("glpk")
results = solver.solve(instance, tee=True, keepfiles=True)

---

## Proximos bloques (pendientes)

1. **Procesar resultados** - extraer Variables y Params optimizados, escribirlos como `process_results(...)`.

Variables clave disponibles tras ejecutar 1-8:

| variable | significado |
|---|---|
| `csv_dir_raw`, `processing_raw`, `t_raw_seconds` | salida de la generacion sin filtrar |
| `csv_dir_filtered`, `processing_filtered`, `t_filtered_seconds` | copia filtrada de los CSVs |
| `csv_filter_report`, `cmp_df` | reportes del filtrado y verificacion de equivalencia |
| `instance_raw`, `instance_flt` | ConcreteModels independientes |
| `t_build_raw`, `t_build_flt` | tiempos de DataPortal/build_instance |
| `solve_results`, `df_solve` | resultados (status, tiempo, objetivo) por solver |
| `solver_status` | dict {nombre: disponible?} de la celda de diagnostico |
| `model` | ultimo AbstractModel |